# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoyuQre/flyrank_assgn_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

A content page receives a higher action score if it has **not been updated recently**, has **low search engagement**, or has **poor search visibility**. Pages that satisfy more of these conditions are given higher priority for review because they are more likely to experience declining search performance.

### Reason Codes

* **STALE_CONTENT** – The page has not been updated for a long time.
* **LOW_CTR** – The page has a relatively low click-through rate.
* **POOR_POSITION** – The page has a poor average search position.
* **LOW_IMPRESSIONS** – The page receives relatively few search impressions.
* **HIGH_PRIORITY** – Multiple warning signals are present, so the page should be reviewed first.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
### Baseline Rule


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df=pd.read_csv('/content/content_refresh_anonymized.csv')
import os
import numpy as np

# Work on a copy
baseline = df.copy()

# -----------------------------
# Rule-based action score
# -----------------------------
baseline["action_score"] = 0
baseline["reason_codes"] = ""

# 1. Stale content
stale = baseline["days_since_last_update"] >= 90
baseline.loc[stale, "action_score"] += 1
baseline.loc[stale, "reason_codes"] += "STALE_CONTENT;"

# 2. Low CTR
low_ctr = baseline["ctr"] < baseline["ctr"].median()
baseline.loc[low_ctr, "action_score"] += 1
baseline.loc[low_ctr, "reason_codes"] += "LOW_CTR;"

# 3. Poor search position
poor_position = baseline["avg_position"] > 20
baseline.loc[poor_position, "action_score"] += 1
baseline.loc[poor_position, "reason_codes"] += "POOR_POSITION;"

# 4. Low impressions
low_impressions = baseline["impressions_90d"] < baseline["impressions_90d"].median()
baseline.loc[low_impressions, "action_score"] += 1
baseline.loc[low_impressions, "reason_codes"] += "LOW_IMPRESSIONS;"

# -----------------------------
# Rank pages
# -----------------------------
baseline = baseline.sort_values(
    by="action_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# -----------------------------
# Save CSV
# -----------------------------
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline.to_csv(output_path, index=False)

print("Saved to:", output_path)

# Preview the highest-priority pages
baseline[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_codes",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].head(20)

Saved to: work/outputs/baseline_action_score.csv


,rank,content_id,action_score,reason_codes,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_6c76f70a8155,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,28,0.0,28.0,104
1,2,content_0dedae17b399,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,59,0.0,45.7,104
2,3,content_e93ec6a94136,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,622,0.0,24.4,104
3,4,content_59cab09d0b52,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,211,0.0,39.2,104
4,5,content_70f58d5042c6,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,398,0.0,31.8,104
5,6,content_494a0d492834,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,218,0.0,34.9,104
6,7,content_d94e4ef1f68c,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,22,0.0,34.6,104
7,8,content_812e9767c288,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,557,0.0,22.8,104
8,9,content_08e6616dbe13,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,483,0.0,46.4,104
9,10,content_956e9b861c22,4,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,22,0.0,70.9,103


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Take the Top 20 pages
top20 = baseline.head(20).copy()

# Recommended action
top20["action"] = "Review and refresh content"

# Confidence based on rule score
top20["confidence"] = top20["action_score"].map({
    4: "High",
    3: "Medium",
    2: "Medium",
    1: "Low",
    0: "Low"
})

# Limitation / uncertainty
top20["what_could_make_it_wrong"] = (
    "Recent external changes, seasonality, or factors not captured in the dataset."
)

# Display
review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_codes",
    "confidence",
    "what_could_make_it_wrong"
]

top20[review_cols]

,rank,content_id,action,reason_codes,confidence,what_could_make_it_wrong
0,1,content_6c76f70a8155,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
1,2,content_0dedae17b399,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
2,3,content_e93ec6a94136,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
3,4,content_59cab09d0b52,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
4,5,content_70f58d5042c6,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
5,6,content_494a0d492834,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
6,7,content_d94e4ef1f68c,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
7,8,content_812e9767c288,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
8,9,content_08e6616dbe13,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."
9,10,content_956e9b861c22,Review and refresh content,STALE_CONTENT;LOW_CTR;POOR_POSITION;LOW_IMPRES...,High,"Recent external changes, seasonality, or facto..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The rule-based baseline is intentionally simple, so some recommendations are likely to be weak. For example, a page may be flagged because it is old or has a low CTR, even though it is performing well overall or has recently become relevant due to seasonal demand. Similarly, pages with naturally low search volume may receive high scores despite not requiring a refresh.

I also verified that no target leakage was introduced into the baseline. The rule does not use `trend_direction` or `trend_pct`, and it excludes identifier fields such as `content_id` and `client_id`. All scoring signals are historical measurements that would be available before making a recommendation, so the baseline reflects information that a content team could realistically use at prediction time.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.